# ♻️ EcoSort — Jalon 1 : Entraînement de l'IA (Recherche & Labo)

**Objectif :** entraîner un modèle de classification des matières d'emballage par
**Transfer Learning (MobileNetV2)** sur le dataset **Garbage Classification** de Kaggle,
puis sauvegarder le modèle pour l'application EcoSort-Search (Jalon 2).

**Dataset :** 6 classes fondamentales — `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`.

**Livrables :**
1. Ce notebook d'entraînement reproductible
2. Le fichier du modèle sauvegardé : `models/ecosort_mobilenetv2.keras`


## 📋 Les 5 catégories officielles de tri

Pour garantir la cohérence du modèle et de l'interface, le système classifie chaque
produit dans l'une des **5 catégories strictes** suivantes :

| Catégorie de Tri | Couleur UI | Types d'emballages & Produits cibles | Matières associées (Dataset) |
| :--- | :--- | :--- | :--- |
| **Poubelle JAUNE** | 🟡 Jaune | Tous les emballages ménagers légers : bouteilles de soda/eau, canettes de boisson, boîtes de conserve, briques de lait, flacons de shampooing, cartons de colis. | `plastic`, `metal`, `cardboard` |
| **Poubelle VERTE** | 🟢 Vert | Uniquement les verres d'emballage : bouteilles de jus ou de vin en verre, pots de confiture, bocaux de conserve. *(Vaisselle cassée interdite).* | `glass` |
| **Poubelle BLEUE** | 🔵 Bleu | Tous les papiers graphiques propres : prospectus publicitaires, journaux, magazines, cahiers, livres, enveloppes. | `paper` |
| **Bac Électronique (D3E)** | 🎛️ Gris | Tout produit fonctionnant avec des piles, une batterie ou une prise électrique : smartphones, écouteurs, chargeurs, mixeurs, montres. | *Cartographié par mots-clés (le dataset ne contient pas de classe électronique)* |
| **Poubelle MARRON / NOIRE** | ⚫ Marron | Déchets résiduels non recyclables : restes alimentaires, emballages plastiques souples (sachets, films), produits d'hygiène, objets multicouches. | `trash` |

> **Architecture de décision :** le CNN prédit la **matière** (6 classes du dataset),
> puis un mapping déterministe `MATERIAL_TO_BIN` attribue la **poubelle**.
> La catégorie **D3E** est gérée en amont par mots-clés sur le nom du produit
> (comme autorisé par le sujet), car le dataset ne contient pas d'électronique.


## 1. Imports et configuration


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import Input, Model, layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print("TensorFlow :", tf.__version__)
print("GPU disponible :", tf.config.list_physical_devices("GPU"))


In [ ]:
# ---------------------------------------------------------------------------
# Configuration générale
# ---------------------------------------------------------------------------
# Sur Kaggle, le dataset "Garbage Classification" ajouté au notebook se trouve
# généralement ici (adaptez si besoin selon le dataset exact ajouté) :
#   /kaggle/input/garbage-classification/Garbage classification/Garbage classification
# En local : placez les 6 sous-dossiers dans data/garbage/
DATA_DIR = Path("data/garbage")                    # <- ADAPTEZ selon votre environnement
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / "ecosort_mobilenetv2.keras"

IMG_SIZE = (224, 224)      # taille native de MobileNetV2
BATCH_SIZE = 32
EPOCHS_HEAD = 5           # phase 1 : tête seule (base gelée)
EPOCHS_FINE = 10           # phase 2 : fine-tuning léger
SEED = 42

# Mapping officiel matière -> poubelle (identique à app/config.py)
MATERIAL_TO_BIN = {
    "plastic": "JAUNE",
    "metal": "JAUNE",
    "cardboard": "JAUNE",
    "glass": "VERTE",
    "paper": "BLEUE",
    "trash": "MARRON",
}
BIN_EMOJI = {"JAUNE": "🟡", "VERTE": "🟢", "BLEUE": "🔵", "MARRON": "⚫", "D3E": "🎛️"}


## 2. Chargement des données

Split **80 % entraînement / 20 % validation** automatique. Les labels sont en
one-hot (`categorical`) car il s'agit d'une classification **multi-classes**.
L'ordre des classes retourné est **alphabétique** — il doit correspondre
exactement à `MODEL_CLASSES` dans `app/config.py`.


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Classes ({num_classes}) : {class_names}")
# Attendu : ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


### Aperçu du dataset, avec la poubelle correspondante


In [ ]:
plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        material = class_names[int(np.argmax(labels[i]))]
        bin_key = MATERIAL_TO_BIN[material]
        plt.title(f"{material} -> {BIN_EMOJI[bin_key]} {bin_key}")
        plt.axis("off")
plt.tight_layout()
plt.show()


## 3. Augmentation de données

Le dataset est petit (~2 500 images), l'augmentation est donc essentielle pour
éviter le surapprentissage. Ces couches ne sont actives qu'à l'entraînement.


In [ ]:
augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.1),
    ],
    name="augmentation",
)


## 4. Modèle : Transfer Learning MobileNetV2

**Point d'architecture important :** le préprocessing MobileNetV2 (normalisation
[-1, 1]) est intégré **dans** le modèle via une couche `Lambda`. L'application
(Jalon 2) peut ainsi envoyer des pixels bruts 0-255 sans dupliquer la logique
de préprocessing — c'est exactement ce qu'attend `app/classifier.py`.


In [ ]:
inputs = Input(shape=(*IMG_SIZE, 3))
x = augmentation(inputs)
x = layers.Lambda(preprocess_input, name="preprocess")(x)

base_model = MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False   # phase 1 : extracteur de features gelé

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = Model(inputs, outputs, name="ecosort_mobilenetv2")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


## 5. Phase 1 — Entraînement de la tête (base gelée)


In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    ModelCheckpoint(str(MODEL_PATH), monitor="val_accuracy", save_best_only=True),
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
)


## 6. Phase 2 — Fine-tuning

On dégèle les 30 dernières couches de MobileNetV2 avec un learning rate très
faible (1e-5) pour spécialiser les features de haut niveau sur nos matières,
sans détruire les poids ImageNet.


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=callbacks,
)


### Courbes d'apprentissage


In [ ]:
acc = history_head.history["accuracy"] + history_fine.history["accuracy"]
val_acc = history_head.history["val_accuracy"] + history_fine.history["val_accuracy"]
loss = history_head.history["loss"] + history_fine.history["loss"]
val_loss = history_head.history["val_loss"] + history_fine.history["val_loss"]
split = len(history_head.history["accuracy"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(acc, label="train"); ax1.plot(val_acc, label="validation")
ax1.axvline(split - 0.5, color="gray", ls="--", label="début fine-tuning")
ax1.set_title("Accuracy"); ax1.legend(); ax1.grid(alpha=.3)
ax2.plot(loss, label="train"); ax2.plot(val_loss, label="validation")
ax2.axvline(split - 0.5, color="gray", ls="--")
ax2.set_title("Loss"); ax2.legend(); ax2.grid(alpha=.3)
plt.show()


## 7. Évaluation — matières ET poubelles

On évalue à deux niveaux :
1. **Précision matière** (les 6 classes du CNN)
2. **Précision poubelle** (après mapping vers les 5 catégories officielles) —
c'est la métrique qui compte pour l'utilisateur final. Elle est mécaniquement
meilleure : confondre `plastic` et `metal` reste un tri correct (les deux vont en JAUNE).


In [ ]:
# Prédictions sur tout le set de validation
y_true, y_pred = [], []
for images, labels in val_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(probs, axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

acc_material = float((y_true == y_pred).mean())
print(f"Précision MATIÈRE  (6 classes)   : {acc_material:.1%}")

# Mapping vers les poubelles
bins_true = np.array([MATERIAL_TO_BIN[class_names[i]] for i in y_true])
bins_pred = np.array([MATERIAL_TO_BIN[class_names[i]] for i in y_pred])
acc_bin = float((bins_true == bins_pred).mean())
print(f"Précision POUBELLE (5 catégories) : {acc_bin:.1%}")


In [ ]:
# Matrice de confusion sur les matières
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

print(classification_report(y_true, y_pred, target_names=class_names))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=class_names, ax=ax1, colorbar=False,
    xticks_rotation=45,
)
ax1.set_title("Confusion — matières (6 classes)")

bin_labels = ["JAUNE", "VERTE", "BLEUE", "MARRON"]
ConfusionMatrixDisplay.from_predictions(
    bins_true, bins_pred, labels=bin_labels, ax=ax2, colorbar=False,
)
ax2.set_title("Confusion — poubelles (mapping officiel)")
plt.tight_layout()
plt.show()


## 8. Test de bout en bout : la chaîne complète de décision

Simulation de la logique exacte de l'application (`app/classifier.py`) :
1. Mots-clés D3E sur le **nom** du produit → priorité au Bac Électronique
2. Sinon, CNN sur l'**image** → matière → poubelle via `MATERIAL_TO_BIN`


In [ ]:
D3E_KEYWORDS = ["pile", "batterie", "chargeur", "usb", "smartphone", "téléphone",
                "écouteur", "bluetooth", "casque", "montre connectée", "électrique",
                "électronique", "tv", "laptop", "ordinateur"]

def is_electronic(product_name: str) -> bool:
    name = product_name.lower()
    return any(kw in name for kw in D3E_KEYWORDS)

def classify_like_the_app(product_name: str, image_tensor=None) -> str:
    """Reproduit la décision de app/classifier.py."""
    if is_electronic(product_name):
        return "D3E", None, 1.0
    if image_tensor is None:
        return "MARRON", None, 0.0     # fallback prudent
    probs = model.predict(image_tensor[None, ...], verbose=0)[0]
    material = class_names[int(np.argmax(probs))]
    return MATERIAL_TO_BIN[material], material, float(probs.max())

# Démonstration sur quelques images de validation + noms fictifs
demo_names = ["Bouteille d'eau minérale 1.5L", "Écouteurs Bluetooth TWS",
              "Pot de confiture fraise", "Journal quotidien"]
for images, labels in val_ds.take(1):
    for name, img in zip(demo_names, images[:4]):
        bin_key, material, conf = classify_like_the_app(name, img)
        detail = f"matière={material} ({conf:.0%})" if material else "mots-clés D3E"
        print(f"{BIN_EMOJI[bin_key]} {bin_key:6} | {detail:28} | {name}")


## 9. Sauvegarde du livrable

Le modèle est sauvegardé au format `.keras` — le fichier attendu par
`app/config.py` (`MODEL_PATH`). Sur Kaggle/Colab, téléchargez-le ensuite et
placez-le dans `models/` à la racine du projet EcoSort-Search.


In [ ]:
model.save(MODEL_PATH)
print(f"✅ Modèle sauvegardé : {MODEL_PATH.resolve()}")
print(f"   Ordre des classes : {class_names}")
print("   -> Doit être identique à MODEL_CLASSES dans app/config.py")

# Sur Google Colab, décommentez pour télécharger directement :
# from google.colab import files
# files.download(str(MODEL_PATH))
